In [4]:
pip install requests pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [6]:
def main():
    print("1. Requisitando dados da API pública da SpaceX (Mirror IBM)...")
    
    # URL estática oficial fornecida pelo curso da IBM para evitar falha de conexão na API
    spacex_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json"
    
    response = requests.get(spacex_url)
    
    if response.status_code != 200:
        print(f"Erro ao acessar os dados! Código de status: {response.status_code}")
        return

    # Converte o JSON em DataFrame
    data = pd.json_normalize(response.json())
    
    # ... resto do código continua exatamente igual ...

In [8]:
import pandas as pd
import numpy as np
import datetime

# Exibir todas as colunas no VSCode
pd.set_option('display.max_columns', None)

def main():
    print("1. Baixando o dataset processado da SpaceX (Mirror IBM/Coursera)...")
    
    # Dataset pré-processado pela IBM para o laboratório de API
    dataset_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_1.csv"
    
    try:
        df = pd.read_csv(dataset_url)
        print("-> Dados carregados com sucesso!")
    except Exception as e:
        print(f"Erro ao carregar o dataset: {e}")
        return

    print("\n2. Informações Gerais do Dataset:")
    print(f"Total de Registros: {df.shape[0]}")
    print(f"Total de Colunas: {df.shape[1]}")

    print("\n3. Primeiras 5 linhas:")
    print(df.head())

    print("\n4. Verificação de Nulos por coluna:")
    print(df.isnull().sum())

    # Salva o arquivo CSV localmente no diretório do seu VSCode
    df.to_csv("dataset_part_1.csv", index=False)
    print("\n-> Arquivo 'dataset_part_1.csv' salvo com sucesso na sua pasta local!")

if __name__ == "__main__":
    main()

1. Baixando o dataset processado da SpaceX (Mirror IBM/Coursera)...
-> Dados carregados com sucesso!

2. Informações Gerais do Dataset:
Total de Registros: 90
Total de Colunas: 17

3. Primeiras 5 linhas:
   FlightNumber        Date BoosterVersion  PayloadMass Orbit    LaunchSite  \
0             1  2010-06-04       Falcon 9  6104.959412   LEO  CCAFS SLC 40   
1             2  2012-05-22       Falcon 9   525.000000   LEO  CCAFS SLC 40   
2             3  2013-03-01       Falcon 9   677.000000   ISS  CCAFS SLC 40   
3             4  2013-09-29       Falcon 9   500.000000    PO   VAFB SLC 4E   
4             5  2013-12-03       Falcon 9  3170.000000   GTO  CCAFS SLC 40   

       Outcome  Flights  GridFins  Reused   Legs LandingPad  Block  \
0    None None        1     False   False  False        NaN    1.0   
1    None None        1     False   False  False        NaN    1.0   
2    None None        1     False   False  False        NaN    1.0   
3  False Ocean        1     False   False

In [10]:
import sqlite3
import pandas as pd

# 1. Carregar o CSV estático específico do laboratório de SQL da IBM
# Esse dataset contém as colunas 'Customer', 'Landing_Outcome', 'Booster_Version', etc.
dataset_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv"
df = pd.read_csv(dataset_url)

# 2. Criar conexão SQLite em memória
conn = sqlite3.connect(":memory:")

# 3. Exportar o DataFrame para a tabela SPACEXTBL
df.to_sql("SPACEXTBL", conn, if_exists="replace", index=False)

def run_query(query_num, description, query):
    print(f"\n--- QUERY {query_num}: {description} ---")
    result = pd.read_sql_query(query, conn)
    print(result)
    return result

# --- QUERIES AJUSTADAS PARA O SCHEMA DO LAB SQL ---

# Query 1: Nomes dos locais de lançamento únicos
run_query(1, "Unique Launch Sites", 
          "SELECT DISTINCT Launch_Site FROM SPACEXTBL")

# Query 2: 5 registros onde o local de lançamento começa com 'CCA'
run_query(2, "Launch Sites Starting with 'CCA'", 
          "SELECT * FROM SPACEXTBL WHERE Launch_Site LIKE 'CCA%' LIMIT 5")

# Query 3: Carga útil total carregada por boosters da NASA (CRS)
run_query(3, "Total Payload Mass for NASA (CRS)", 
          "SELECT SUM(PAYLOAD_MASS__KG_) AS TotalPayloadMass FROM SPACEXTBL WHERE Customer LIKE '%NASA (CRS)%'")

# Query 4: Carga útil média carregada pela versão de booster F9 v1.1
run_query(4, "Average Payload Mass for F9 v1.1", 
          "SELECT AVG(PAYLOAD_MASS__KG_) AS AvgPayloadMass FROM SPACEXTBL WHERE Booster_Version LIKE 'F9 v1.1%'")

# Query 5: Data do primeiro pouso bem-sucedido em plataforma terrestre (Ground Pad)
run_query(5, "First Successful Ground Pad Landing Date", 
          "SELECT MIN(Date) AS FirstSuccessDate FROM SPACEXTBL WHERE Landing_Outcome = 'Success (ground pad)'")

# Query 6: Boosters que pousaram na Drone Ship com carga entre 4000 e 6000 kg
run_query(6, "Successful Drone Ship Landings (Payload 4000-6000kg)", 
          "SELECT Booster_Version FROM SPACEXTBL WHERE Landing_Outcome = 'Success (drone ship)' AND PAYLOAD_MASS__KG_ BETWEEN 4000 AND 6000")

# Query 7: Total de missões bem-sucedidas vs falhas
run_query(7, "Total Success and Failure Outcomes", 
          "SELECT Landing_Outcome, COUNT(*) AS TotalCount FROM SPACEXTBL GROUP BY Landing_Outcome")

# Query 8: Boosters que carregaram a massa de carga útil máxima
run_query(8, "Boosters with Maximum Payload Mass", 
          "SELECT Booster_Version, PAYLOAD_MASS__KG_ FROM SPACEXTBL WHERE PAYLOAD_MASS__KG_ = (SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTBL)")

# Query 9: Registros de falhas em Drone Ship no ano de 2015
run_query(9, "2015 Failed Drone Ship Landings", 
          "SELECT Date, Booster_Version, Launch_Site FROM SPACEXTBL WHERE Landing_Outcome = 'Failure (drone ship)' AND Date LIKE '2015%'")

# Query 10: Rank de contagem de resultados de pouso entre 2010-06-04 e 2017-03-20 em ordem decrescente
run_query(10, "Rank Landing Outcomes Between 2010-06-04 and 2017-03-20", 
          "SELECT Landing_Outcome, COUNT(*) AS Count FROM SPACEXTBL WHERE Date BETWEEN '2010-06-04' AND '2017-03-20' GROUP BY Landing_Outcome ORDER BY Count DESC")

# Fechar conexão
conn.close()


--- QUERY 1: Unique Launch Sites ---
    Launch_Site
0   CCAFS LC-40
1   VAFB SLC-4E
2    KSC LC-39A
3  CCAFS SLC-40

--- QUERY 2: Launch Sites Starting with 'CCA' ---
         Date Time (UTC) Booster_Version  Launch_Site  \
0  2010-06-04   18:45:00  F9 v1.0  B0003  CCAFS LC-40   
1  2010-12-08   15:43:00  F9 v1.0  B0004  CCAFS LC-40   
2  2012-05-22    7:44:00  F9 v1.0  B0005  CCAFS LC-40   
3  2012-10-08    0:35:00  F9 v1.0  B0006  CCAFS LC-40   
4  2013-03-01   15:10:00  F9 v1.0  B0007  CCAFS LC-40   

                                             Payload  PAYLOAD_MASS__KG_  \
0               Dragon Spacecraft Qualification Unit                  0   
1  Dragon demo flight C1, two CubeSats, barrel of...                  0   
2                              Dragon demo flight C2                525   
3                                       SpaceX CRS-1                500   
4                                       SpaceX CRS-2                677   

       Orbit         Customer Mission